In [2]:
import pandas as pd
import openpyxl
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [3]:
raw=pd.read_excel("/workspaces/Customer_Segmentation_KMeans_Clustering/data/raw/Online Retail.xlsx")
raw.head(2)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
print('shape before wrangling:', raw.shape)

shape before wrangling: (541909, 8)


In [5]:
from src.data.cleaner import DataCleaner
cleaner=DataCleaner(raw)
df=cleaner.wrangle()
print('shape after wrangling:', df.shape)

2026-08-07 21:39:59,552 - INFO - Starting data cleaning and feature engineering
2026-08-07 21:40:00,264 - INFO - Data cleaning completed. Customer df shape: (4239, 7)


shape after wrangling: (4239, 7)


In [6]:
print('Raw shape after running cleaner:', raw.shape)

Raw shape after running cleaner: (541909, 8)


In [7]:
from src.data.loader import DataLoader
path="/workspaces/Customer_Segmentation_KMeans_Clustering/data/raw/Online Retail.xlsx"
loader=DataLoader(path)
df=loader.load_data()
df.shape

2026-08-07 21:40:00,575 - INFO - Loading raw data from: /workspaces/Customer_Segmentation_KMeans_Clustering/data/raw/Online Retail.xlsx
2026-08-07 21:40:44,757 - INFO - Raw data successfully loaded. Shape: (541909, 8)


(541909, 8)

In [8]:
df.head(2)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [9]:
from src.data.validator import DataValidator
validator=DataValidator(df)
validator.validate_columns()

2026-08-07 21:40:45,048 - INFO - Validating raw data columns
2026-08-07 21:40:45,049 - INFO - Raw data validation successful


In [10]:
from src.data.loader import DataLoader
from src.data.validator import DataValidator
from src.data.cleaner import DataCleaner

path='/workspaces/Customer_Segmentation_KMeans_Clustering/data/raw/Online Retail.xlsx'
loader=DataLoader(path)
df=loader.load_data()
validator=DataValidator(df)
validator.validate_columns()
print('Raw data columns:', df.columns.to_list())
print('Raw data shape:', df.shape)
cleaner=DataCleaner(df)
customer_df=cleaner.wrangle()
print('Clean customer dataframe columns:', customer_df.columns.to_list())
print('Customer data shape:', customer_df.shape)


2026-08-07 21:40:45,313 - INFO - Loading raw data from: /workspaces/Customer_Segmentation_KMeans_Clustering/data/raw/Online Retail.xlsx
2026-08-07 21:41:29,538 - INFO - Raw data successfully loaded. Shape: (541909, 8)
2026-08-07 21:41:29,560 - INFO - Validating raw data columns
2026-08-07 21:41:29,561 - INFO - Raw data validation successful
2026-08-07 21:41:29,630 - INFO - Starting data cleaning and feature engineering


Raw data columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
Raw data shape: (541909, 8)


2026-08-07 21:41:30,256 - INFO - Data cleaning completed. Customer df shape: (4239, 7)


Clean customer dataframe columns: ['CustomerID', 'TotalSpent', 'Orders', 'ItemsPurchased', 'CountriesPurchased', 'AvgOrderValue', 'Recency']
Customer data shape: (4239, 7)


In [11]:
from src.features.preprocessor import DataPreprocessor
print('Input shape:', customer_df.shape)
preprocessor=DataPreprocessor(customer_df)
X=preprocessor.preprocess()
print('Output shape:', X.shape)



2026-08-07 21:41:30,307 - INFO - Starting feature preprocessing
2026-08-07 21:41:30,311 - INFO - Feature preprocessing completed. Input shape: (4239, 7), Output shape: (4239, 5)


Input shape: (4239, 7)
Output shape: (4239, 5)


In [12]:
from src.models.k_means_model import KMeansModel
KMeansobj=KMeansModel(X)
model=KMeansobj.build_model()
print(model)

2026-08-07 21:41:33,152 - INFO - Starting KMeans model training. Training data shape (4239, 5)
2026-08-07 21:41:33,341 - INFO - KMeans model training completed


Pipeline(steps=[('standardscaler', StandardScaler()),
                ('kmeans', KMeans(n_clusters=5, random_state=42))])


In [13]:
from src.models.cluster_assigner import ClusterAssigner

In [14]:
cluster_assigner=ClusterAssigner(customer_df=customer_df, X=X, model=model)
clusters_df=cluster_assigner.assign_clusters()
print('Clusters shape:', clusters_df.shape)
clusters_df.head()


2026-08-07 21:41:33,753 - INFO - Assigning clusters to customers
2026-08-07 21:41:33,758 - INFO - Cluster assignment completed. Customers clustered: 4239


Clusters shape: (4239, 8)


,CustomerID,TotalSpent,Orders,ItemsPurchased,CountriesPurchased,AvgOrderValue,Recency,Cluster
0,12347.0,4310.00,7,2458,1,615.714286,1,2
1,12348.0,1797.24,4,2341,1,449.310000,74,0
2,12349.0,1757.55,1,631,1,1757.550000,18,0
3,12350.0,334.40,1,197,1,334.400000,309,1
4,12352.0,1240.73,4,380,1,310.182500,35,3


In [15]:
import joblib
from src.models.model_manager import ModelManager

In [16]:
manager = ModelManager()
loaded_model = manager.load_model()

print(loaded_model)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('kmeans', KMeans(n_clusters=5, random_state=42))])


In [17]:
from src.config.config import (
    BASE_DIR,
    RAW_DATA_PATH,
    MODEL_PATH,
    N_CLUSTERS,
    RANDOM_STATE
)
print(BASE_DIR)
print(RAW_DATA_PATH)
print(MODEL_PATH)
print(N_CLUSTERS)
print(RANDOM_STATE)


ImportError: cannot import name 'N_CLUSTERS' from 'src.config.config' (/workspaces/Customer_Segmentation_KMeans_Clustering/src/config/config.py)

In [ ]:
from src.utils.logger import logger

logger.info("Testing logger")

2026-08-07 12:11:30,264 - INFO - Testing logger


In [18]:
from src.utils.logger import logger

logger.info("Testing file logging")

2026-08-07 21:42:42,011 - INFO - Testing file logging


In [3]:
from src.utils.logger import logger

logger.info("Testing file logging")

2026-08-07 21:58:01,748 - INFO - Testing file logging
